# 13 · Robustez: campo de estudio (educación superior)

La ocupación capta *dónde* trabaja una persona, pero no *qué* estudió: dos dimensiones de capital humano que no coinciden por completo. Parada-Contzen y Jara (2025) documentan para Chile brechas heterogéneas por **campo de estudio** entre trabajadores con educación superior. Este notebook incorpora el campo de estudio (`cinef13_area`, clasificación CINE-F 2013, cobertura total entre ocupados con educación superior) para verificar si la brecha dentro de la ocupación exacta sobrevive al controlar además por el área de formación — dialogando directamente con esa literatura.

In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

os.makedirs('outputs/data', exist_ok=True)
RUTA_CASEN = '../../CASEN'
COLS = ['sexo','edad','e6a','o10','oficio4_08','ytrabajocor','expr','activ','varunit','cinef13_area']
frames = []
for anio in [2022, 2024]:
    df = pd.read_stata(f'{RUTA_CASEN}/casen_{anio}.dta', columns=COLS, convert_categoricals=False)
    df['anio'] = anio
    frames.append(df)
panel = pd.concat(frames, ignore_index=True)

CAMPO = {1:'Educación', 2:'Artes y humanidades', 3:'Ciencias sociales/periodismo',
         4:'Administración/negocios/derecho', 5:'Ciencias naturales/matemáticas',
         6:'TIC', 7:'Ingeniería/industria/construcción', 8:'Agricultura/veterinaria',
         9:'Salud y bienestar', 10:'Servicios'}

m = panel[(panel['activ']==1)&(panel['ytrabajocor']>0)&(panel['oficio4_08'].notna())&
          (panel['e6a']>=12)].copy()                        # educación superior
m = m[(m['o10']>0)&(m['o10']<=112)]
m = m[m['cinef13_area'].between(1,10)].copy()               # campo de estudio válido
NIVEL_EDUC = {12:'Técnica sup.',13:'Universitaria',14:'Posgrado',15:'Posgrado'}
m['mujer'] = (m['sexo']==2).astype(int)
m['nivel_grp'] = pd.Categorical(m['e6a'].map(NIVEL_EDUC))
m['log_ingreso'] = np.log(m['ytrabajocor']); m['edad2'] = m['edad']**2
m['anio'] = pd.Categorical(m['anio'])
m['campo'] = pd.Categorical(m['cinef13_area'].map(CAMPO))
conteo = m['oficio4_08'].value_counts(); val = set(conteo[conteo>=30].index)
m['oficio4_grp'] = pd.Categorical(m['oficio4_08'].apply(lambda c: str(int(c)) if c in val else 'otras'))
m['cluster'] = m['anio'].astype(str) + '_' + m['varunit'].astype(int).astype(str)
print(f'Submuestra con educación superior y campo de estudio válido: {len(m):,}')

Submuestra con educación superior y campo de estudio válido: 69,956


## 1. La brecha al agregar campo de estudio y ocupación

In [2]:
base = 'log_ingreso ~ mujer + edad + edad2 + C(nivel_grp) + o10 + C(anio)'
especs = [
    ('Sin campo ni ocupación',        base),
    ('+ Campo de estudio (10 áreas)',  base + ' + C(campo)'),
    ('+ Ocupación (4 dígitos)',        base + ' + C(oficio4_grp)'),
    ('+ Campo Y ocupación',            base + ' + C(campo) + C(oficio4_grp)'),
]
filas = []
for nombre, f in especs:
    r = smf.wls(f, data=m, weights=m['expr']).fit(cov_type='cluster', cov_kwds={'groups': m['cluster']})
    b = r.params['mujer']; ic = r.conf_int().loc['mujer']
    filas.append({'especificación': nombre, 'brecha_pct': (np.exp(b)-1)*100,
                  'ic95_inf': (np.exp(ic[0])-1)*100, 'ic95_sup': (np.exp(ic[1])-1)*100})
tab = pd.DataFrame(filas)
tab['IC95%'] = tab.apply(lambda r: f'[{r.ic95_inf:.1f}, {r.ic95_sup:.1f}]', axis=1)
print(tab[['especificación','brecha_pct','IC95%']].round(2).to_string(index=False))
tab.to_csv('outputs/data/robustez_campo_estudio.csv', index=False, encoding='utf-8-sig')

               especificación  brecha_pct          IC95%
       Sin campo ni ocupación      -17.08 [-18.3, -15.8]
+ Campo de estudio (10 áreas)      -13.74 [-15.2, -12.2]
      + Ocupación (4 dígitos)      -11.53 [-12.9, -10.1]
          + Campo Y ocupación      -10.43  [-11.9, -9.0]


### Interpretación

Entre trabajadores con educación superior, el campo de estudio explica una parte real de la brecha —de -17.1% (sin campo ni ocupación) a -13.7% agregando solo el área de formación—, coherente con Parada-Contzen y Jara (2025). Pero su aporte es **parcialmente redundante con la ocupación**: la ocupación a 4 dígitos por sí sola la lleva a -11.5%, y agregar el campo de estudio *encima* de la ocupación exacta solo suma ~1 punto adicional (**-10.4%**). Es decir, buena parte de lo que el campo de estudio captaría opera vía la ocupación en la que efectivamente se trabaja.

El punto para la publicación: **incluso comparando profesionales del mismo nivel educativo, el mismo campo de estudio y la misma ocupación exacta, persiste una brecha de ~10%** entre hombres y mujeres. El área de formación —el argumento de "estudiaron cosas distintas"— no rescata la explicación composicional del diferencial.